# Eval 4 — Adapter Cooldown Effect on Robot Command Rate

**Target**: `robot/bittle_adapter.py` — `INTENT_COOLDOWN_SEC = 2.5`  
**Robot needed**: Simulated (no physical robot required)  
**Reruns model**: No (replay-based)

Compares `INTENT_COOLDOWN_SEC=0.0` vs `2.5` (production) under two timing modes:
- `online_mode`: 1 window every 3.0 s (real-time processing)  
- `offline_mode`: 1 window every 0.5 s (fast-replay, faster than real-time)

Key figure (Cell 3): the adapter cooldown matters **most** in offline mode,
where the pipeline can dispatch much faster than the robot can execute.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, os.path.abspath('..'))
from eval_utils import INTENT_COLORS

results = pd.read_csv('results.csv')
print(f"Loaded {len(results)} rows")
print(f"Conditions:    {results['condition'].unique().tolist()}")
print(f"Timing modes:  {results['timing_mode'].unique().tolist()}")
results.head(8)

## Cell 2 — Summary table: mean command_rate and drop rates

In [ ]:
summary = (
    results
    .groupby(['condition', 'timing_mode'])[
        ['command_rate', 'dedup_drop_rate', 'cooldown_drop_rate', 'n_commands_sent']
    ]
    .agg(['mean', 'std'])
    .round(3)
)
print("Mean ± std across 30 videos:")
summary

## Cell 3 — KEY FIGURE: command_rate by condition × timing mode

Shows that the adapter cooldown matters most in offline mode.

In [ ]:
cond_order   = ['no_cooldown', 'with_cooldown']
timing_order = ['online_mode', 'offline_mode']
cond_labels  = ['No cooldown\n(0.0 s)', 'With cooldown\n(2.5 s)']
timing_colors = {'online_mode': '#90C8E8', 'offline_mode': '#D85A30'}

# Pivot: mean command_rate per (condition, timing_mode)
pivot_mean = (
    results.groupby(['condition', 'timing_mode'])['command_rate']
    .mean()
    .unstack('timing_mode')
    .reindex(index=cond_order, columns=timing_order)
)
pivot_std = (
    results.groupby(['condition', 'timing_mode'])['command_rate']
    .std()
    .unstack('timing_mode')
    .reindex(index=cond_order, columns=timing_order)
)

x = np.arange(len(cond_order))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))

for i, (timing, color) in enumerate(timing_colors.items()):
    offset = (i - 0.5) * width
    bars = ax.bar(x + offset, pivot_mean[timing], width,
                  yerr=pivot_std[timing], capsize=5,
                  color=color, edgecolor='k',
                  label=timing.replace('_', ' '))
    for bar, val in zip(bars, pivot_mean[timing]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(cond_labels, fontsize=11)
ax.set_ylabel('Mean command rate  (commands / min simulated time)')
ax.set_title('Robot command rate: adapter cooldown ablation\n(mean ± std across 30 videos)')
ax.legend(title='Timing mode', fontsize=10)
plt.tight_layout()
plt.show()

## Cell 4 — Stacked bar chart: proportion fired vs dropped per condition

In [ ]:
# For each (condition, timing_mode), compute mean proportions across videos
results['sent_rate']         = results['n_commands_sent']    / results['n_windows']
results['total_dedup_rate']  = results['dedup_drop_rate']
results['total_cooldown_rate'] = results['cooldown_drop_rate']

group_keys = ['condition', 'timing_mode']
stack_cols  = ['sent_rate', 'total_dedup_rate', 'total_cooldown_rate']
agg = results.groupby(group_keys)[stack_cols].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

stack_colors = ['#5DCAA5', '#B4B2A9', '#EF9F27']
stack_labels = ['Fired (sent)', 'Dropped: dedup', 'Dropped: cooldown']

for ax, timing in zip(axes, timing_order):
    sub = agg.xs(timing, level='timing_mode').reindex(cond_order)
    bottom = np.zeros(len(cond_order))
    for col, color, label in zip(stack_cols, stack_colors, stack_labels):
        vals = sub[col].values
        ax.bar(cond_order, vals, bottom=bottom, color=color,
               label=label, edgecolor='k', width=0.5)
        bottom += vals
    ax.set_title(timing.replace('_', ' '), fontsize=11)
    ax.set_ylabel('Fraction of windows')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=10)

handles = [mpatches.Patch(color=c, label=l)
           for c, l in zip(stack_colors, stack_labels)]
fig.legend(handles=handles, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.1), fontsize=10)
fig.suptitle('Window fate: fired vs dropped per condition', fontsize=12)
plt.tight_layout()
plt.show()

## Cell 5 — Per-window fired/dropped line chart for one video (offline mode)

In [ ]:
# Re-simulate one representative video to get per-window fired flags
sys.path.insert(0, os.path.abspath('../..'))
from eval_utils import load_all_sessions

sessions = load_all_sessions('../session_csvs')

if sessions.empty:
    print("No session CSVs found — generate them first.")
else:
    # Pick representative video (most windows)
    rep_video = sessions.groupby('video')['window_index'].count().idxmax()
    vid_df = sessions[sessions['video'] == rep_video].sort_values('window_index')
    intents = vid_df['effective_intent'].tolist()
    windows = list(range(len(intents)))

    def simulate_per_window(intents_list, cooldown_sec, window_interval):
        """Return list of 1 (fired) / 0 (dropped) per window."""
        current_pose = None
        last_intent_time = -float('inf')
        sim_time = 0.0
        fired = []
        for intent in intents_list:
            intent = intent.upper()
            if intent == current_pose:
                fired.append(0)
            elif (sim_time - last_intent_time) < cooldown_sec:
                fired.append(0)
            else:
                fired.append(1)
                current_pose = intent
                last_intent_time = sim_time
            sim_time += window_interval
        return fired

    OFFLINE_INTERVAL = 0.5
    no_cd_fired   = simulate_per_window(intents, 0.0,  OFFLINE_INTERVAL)
    with_cd_fired = simulate_per_window(intents, 2.5, OFFLINE_INTERVAL)

    # Jitter for visibility
    jitter = 0.08
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(windows, [v + jitter for v in no_cd_fired],
            'o-', color='#E8A090', markersize=5, linewidth=1,
            label='no_cooldown (0.0 s)', alpha=0.85)
    ax.plot(windows, [v - jitter for v in with_cd_fired],
            's-', color='#90C8E8', markersize=5, linewidth=1,
            label='with_cooldown (2.5 s)', alpha=0.85)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['dropped', 'fired'])
    ax.set_xlabel('Window index')
    ax.set_title(f'Per-window dispatch — offline mode (0.5 s/window) — {rep_video}')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

## Cell 6 — Interpretation

### Why offline mode makes the adapter cooldown more important

In **online mode** (3.0 s/window), windows arrive every 3.0 s.  The adapter
cooldown of 2.5 s is shorter than the window interval, so most intents pass
through the cooldown gate and are dispatched.  Deduplication handles the bulk
of drops (same intent sustained across windows).

In **offline mode** (0.5 s/window), the pipeline replays a video faster than
real-time.  Windows arrive every 0.5 s — six times faster than the 2.5 s
cooldown.  Without the cooldown (`no_cooldown`), up to 6× more commands per
second are dispatched than the robot can physically execute.  **This would
cause servo thrashing**: the robot receives a new motor command while still
completing the previous motion, causing abrupt direction reversals and
potential hardware damage.

The `with_cooldown` condition shows that the 2.5 s gate brings the offline
command rate close to the online rate — the robot effectively sees the same
physical command cadence regardless of pipeline speed.

### Why 2.5 s was chosen

The longest single motion budget in `_INTENT_COMMANDS` is **ENGAGE = 3.5 s**
(the `khi` wave).  A cooldown of 2.5 s ensures at most one command per
physical motion cycle for all but the ENGAGE motion — the robot will
complete its movement before the next distinct intent is considered.
For ENGAGE specifically, the 2.5 s cooldown means the next command *could*
arrive 1 s before the wave finishes, but the `_executing` flag in the
real adapter prevents dispatch while a motion is running.

### Limitation

This simulation does not model the `_executing` flag or the threading queue
in `BittleXAdapter`.  Real-hardware command rates may differ slightly.  The
key insight (cooldown prevents burst dispatch in offline mode) holds regardless.